# `specialize()` — Partial Evaluation Proof-of-Concept

This notebook demonstrates that `specialize()` triggers **real structural code transformations** in the Finch compiler — not just currying / value embedding.

## Background

The Finch compiler represents tensor metadata in two kinds of IR nodes:

| IR node | Example | Properties |
|---|---|---|
| `value(sym, T)` | `value(:A_lvl_stop, Int64)` | Opaque to the prover — represents a runtime variable |
| `literal(v)` | `literal(100)` | Transparent to the prover — participates in constant folding |

Both are `isconstant` (they don't depend on loop indices), but only `literal` nodes let the compiler's `prove()` and `simplify()` passes resolve comparisons and fold expressions at compile time.

When we call `specialize(tensor)`, the Dense level shapes are lifted into the type domain and virtualized as `literal(N)` instead of `value(:stop, Ti)`. This lets the prover resolve conditions like `measure(ext) == 1` (i.e., `stop - start + 1 == 1`), which triggers structural optimizations in the lowering passes.

### The key prove site

In `src/looplets/lookups.jl`, the `LookupStyle` lowerer checks:

```julia
if prove(ctx, call(==, measure(root.ext.val), target))
    # emit direct assignment: idx = start (no loop)
else
    # emit for-loop: for idx = start:stop
end
```

When a Dense level has shape 1 and that shape is a `literal(1)`, `measure = literal(1) - literal(1) + 1 = literal(1)`, and `prove(ctx, literal(1) == 1)` returns `true`. The for-loop is eliminated entirely. With `value(:stop)`, the prover returns `false` (can't resolve) and the loop is emitted.

In [1]:
using Pkg
Pkg.activate(joinpath(@__DIR__, "..", ".."))
using Finch

  Activating 

project at `~/Finch.jl`


## Test 1: Loop Elimination — `Dense(Dense(Element))`, 4×1

A fully dense 4×1 matrix has outer Dense shape = 1 and inner Dense shape = 4.
With specialization:
- The outer `for j = 1:1` loop should **vanish** (proved `measure == 1`)
- The inner loop bound should become a **literal** `1:4` instead of a runtime variable `1:A_lvl_2_stop`

In [2]:
A_dense = Tensor(Dense(Dense(Element(0.0))), rand(4, 1))
s = Scalar(0.0)

println("Normal generated code:")
println("─"^60)
println(@finch_code begin
    s .= 0
    for j in _, i in _
        s[] += A_dense[i, j]
    end
    return s
end)

Normal generated code:


────────────────────────────────────────────────────────────
begin


    s_data = ((ex.bodies[1]).bodies[1]).tns.bind
    A_dense_lvl = ((ex.bodies[1]).bodies[2]).body.body.rhs.tns.bind.lvl
    A_dense_lvl_stop = A_dense_lvl.shape
    A_dense_lvl_2 = A_dense_lvl.lvl
    A_dense_lvl_2_stop = A_dense_lvl_2.shape
    A_dense_lvl_3 = A_dense_lvl_2.lvl
    A_dense_lvl_3_val = A_dense_lvl_3.val
    s_val = 0
    for j_3 = 1:A_dense_lvl_stop
        A_dense_lvl_q = (1 - 1) * A_dense_lvl_stop + j_3
        for i_3 = 1:A_dense_lvl_2_stop
            A_dense_lvl_2_q = (A_dense_lvl_q - 1) * A_dense_lvl_2_stop + i_3
            A_dense_lvl_3_val_2 = A_dense_lvl_3_val[A_dense_lvl_2_q]
            s_val = A_dense_lvl_3_val_2 + s_val
        end
    end
    s_data.val = s_val
    (s = s_data,)
end


In [3]:
sA_dense = specialize(A_dense)

println("Specialized generated code:")
println("─"^60)
println(@finch_code begin
    s .= 0
    for j in _, i in _
        s[] += sA_dense[i, j]
    end
    return s
end)

Specialized generated code:
────────────────────────────────────────────────────────────
begin


    s_data = ((ex.bodies[1]).bodies[1]).tns.bind
    tns_lvl = ((ex.bodies[1]).bodies[2]).body.body.rhs.tns.bind.body.lvl
    tns_lvl_2 = tns_lvl.lvl
    tns_lvl_3 = tns_lvl_2.lvl
    tns_lvl_3_val = tns_lvl_3.val
    s_val = 0
    tns_lvl_q = (1 - 1) * 1 + 1
    for i_3 = 1:4
        tns_lvl_2_q = (tns_lvl_q - 1) * 4 + i_3
        tns_lvl_3_val_2 = tns_lvl_3_val[tns_lvl_2_q]
        s_val = tns_lvl_3_val_2 + s_val
    end
    s_data.val = s_val
    (s = s_data,)
end


### What changed

| Feature | Normal | Specialized |
|---|---|---|
| Outer loop | `for j_3 = 1:A_lvl_stop` | **eliminated** — `j=1` inlined |
| Inner loop bound | `for i_3 = 1:A_lvl_2_stop` | `for i_3 = 1:4` (literal) |
| Shape variable extraction | `A_lvl_stop = A_lvl.shape` | **gone** (not needed) |

The outer for-loop with 1 iteration was collapsed to a direct assignment by the `LookupStyle` lowerer, because `prove(ctx, measure(ext) == 1)` succeeded with the literal shape.

## Test 2: Shape Variables Eliminated — `Dense(Dense(Element))`, 3×2

Even when the outer loop isn't eliminated (shape > 1), specialization still removes runtime shape variables from the preamble — the shapes are baked into the code as literals.

In [4]:
A_3x2 = Tensor(Dense(Dense(Element(0.0))), rand(3, 2))
s = Scalar(0.0)

normal_code = string(@finch_code begin
    s .= 0
    for j in _, i in _
        s[] += A_3x2[i, j]
    end
    return s
end)

sA_3x2 = specialize(A_3x2)
specialized_code = string(@finch_code begin
    s .= 0
    for j in _, i in _
        s[] += sA_3x2[i, j]
    end
    return s
end)

println("Normal code has _stop variables:     ", occursin("_stop", normal_code))
println("Specialized has tns shape vars:       ", 
    occursin("tns_lvl_stop", specialized_code) || occursin("tns_lvl_2_stop", specialized_code))
println()
println("Normal:")
println(normal_code)
println()
println("Specialized:")
println(specialized_code)

Normal code has _stop variables:     true


Specialized has tns shape vars:       false

Normal:
begin
    s_data = ((ex.bodies[1]).bodies[1]).tns.bind
    A_3x2_lvl = ((ex.bodies[1]).bodies[2]).body.body.rhs.tns.bind.lvl
    A_3x2_lvl_stop = A_3x2_lvl.shape
    A_3x2_lvl_2 = A_3x2_lvl.lvl
    A_3x2_lvl_2_stop = A_3x2_lvl_2.shape
    A_3x2_lvl_3 = A_3x2_lvl_2.lvl
    A_3x2_lvl_3_val = A_3x2_lvl_3.val
    s_val = 0
    for j_3 = 1:A_3x2_lvl_stop
        A_3x2_lvl_q = (1 - 1) * A_3x2_lvl_stop + j_3
        for i_3 = 1:A_3x2_lvl_2_stop
            A_3x2_lvl_2_q = (A_3x2_lvl_q - 1) * A_3x2_lvl_2_stop + i_3
            A_3x2_lvl_3_val_2 = A_3x2_lvl_3_val[A_3x2_lvl_2_q]
            s_val = A_3x2_lvl_3_val_2 + s_val
        end
    end
    s_data.val = s_val
    (s = s_data,)
end

Specialized:
begin
    s_data = ((ex.bodies[1]).bodies[1]).tns.bind
    tns_lvl = ((ex.bodies[1]).bodies[2]).body.body.rhs.tns.bind.body.lvl
    tns_lvl_2 = tns_lvl.lvl
    tns_lvl_3 = tns_lvl_2.lvl
    tns_lvl_3_val = tns_lvl_3.val
    s_val = 0
    for j_3

Notice that in the specialized code:
- No `tns_lvl_stop` or `tns_lvl_2_stop` variables appear in the preamble
- Loop bounds use literal constants (e.g., `1:2`, `1:3`) directly
- Index calculations like `(1 - 1) * 2 + j_3` use literal `2` instead of a runtime variable

## Test 3: SpMV with 1-Column Sparse Matrix — `Dense(SparseList(Element))`, 5×1

This test mixes specialization (on the outer Dense) with a non-specialized inner level (SparseList). The outer column loop should still be eliminated because the Dense shape is `literal(1)`.

In [5]:
A_sparse = Tensor(Dense(SparseList(Element(0.0))), fsprand(5, 1, 0.6))
y = Tensor(Dense(Element(0.0)))

println("Normal generated code:")
println("─"^60)
println(@finch_code begin
    y .= 0
    for j in _, i in _
        y[i] += A_sparse[i, j]
    end
    return y
end)

Normal generated code:


────────────────────────────────────────────────────────────
begin


    y_lvl = ((ex.bodies[1]).bodies[1]).tns.bind.lvl
    y_lvl_2 = y_lvl.lvl
    y_lvl_2_val = y_lvl_2.val
    A_sparse_lvl = ((ex.bodies[1]).bodies[2]).body.body.rhs.tns.bind.lvl
    A_sparse_lvl_stop = A_sparse_lvl.shape
    A_sparse_lvl_2 = A_sparse_lvl.lvl
    A_sparse_lvl_2_ptr = A_sparse_lvl_2.ptr
    A_sparse_lvl_2_idx = A_sparse_lvl_2.idx
    A_sparse_lvl_2_stop = A_sparse_lvl_2.shape
    A_sparse_lvl_3 = A_sparse_lvl_2.lvl
    A_sparse_lvl_3_val = A_sparse_lvl_3.val
    Finch.resize_if_smaller!(y_lvl_2_val, A_sparse_lvl_2_stop)
    Finch.fill_range!(y_lvl_2_val, 0.0, 1, A_sparse_lvl_2_stop)
    for j_3 = 1:A_sparse_lvl_stop
        A_sparse_lvl_q = (1 - 1) * A_sparse_lvl_stop + j_3
        A_sparse_lvl_2_q = A_sparse_lvl_2_ptr[A_sparse_lvl_q]
        A_sparse_lvl_2_q_stop = A_sparse_lvl_2_ptr[A_sparse_lvl_q + 1]
        if A_sparse_lvl_2_q < A_sparse_lvl_2_q_stop
            A_sparse_lvl_2_i1 = A_sparse_lvl_2_idx[A_sparse_lvl_2_q_stop - 1]
        else
            A_sparse_lvl_

){Int64}(ElementLevel{0.0, Float64, Int64}(y_lvl_2_val), A_sparse_lvl_2_stop)),)
end


In [6]:
sA_sparse = specialize(A_sparse)

println("Specialized generated code:")
println("─"^60)
println(@finch_code begin
    y .= 0
    for j in _, i in _
        y[i] += sA_sparse[i, j]
    end
    return y
end)

Specialized generated code:
────────────────────────────────────────────────────────────
begin


    y_lvl = ((ex.bodies[1]).bodies[1]).tns.bind.lvl
    y_lvl_2 = y_lvl.lvl
    y_lvl_2_val = y_lvl_2.val
    tns_lvl = ((ex.bodies[1]).bodies[2]).body.body.rhs.tns.bind.body.lvl
    tns_lvl_2 = tns_lvl.lvl
    tns_lvl_2_ptr = tns_lvl_2.ptr
    tns_lvl_2_idx = tns_lvl_2.idx
    tns_lvl_2_stop = tns_lvl_2.shape
    tns_lvl_3 = tns_lvl_2.lvl
    tns_lvl_3_val = tns_lvl_3.val
    Finch.resize_if_smaller!(y_lvl_2_val, tns_lvl_2_stop)
    Finch.fill_range!(y_lvl_2_val, 0.0, 1, tns_lvl_2_stop)
    tns_lvl_q = (1 - 1) * 1 + 1
    tns_lvl_2_q = tns_lvl_2_ptr[tns_lvl_q]
    tns_lvl_2_q_stop = tns_lvl_2_ptr[tns_lvl_q + 1]
    if tns_lvl_2_q < tns_lvl_2_q_stop
        tns_lvl_2_i1 = tns_lvl_2_idx[tns_lvl_2_q_stop - 1]
    else
        tns_lvl_2_i1 = 0
    end
    phase_stop = min(tns_lvl_2_stop, tns_lvl_2_i1)
    if phase_stop >= 1
        if tns_lvl_2_idx[tns_lvl_2_q] < 1
            tns_lvl_2_q = Finch.scansearch(tns_lvl_2_idx, 1, tns_lvl_2_q, tns_lvl_2_q_stop - 1)
        end
        while tru

The outer `for j = 1:A_lvl_stop` loop is gone in the specialized version — the SparseList stepper code runs directly at top level instead of being wrapped in a column loop.

The inner SparseList structure (`while true` / `phase_stop = min(...)`) remains unchanged because SparseList shapes and indices are still runtime values — only Dense shapes are currently specialized.

## Test 4: Correctness Verification

Verify that specialized execution produces identical results to normal execution.

In [7]:
A_check = Tensor(Dense(Dense(Element(0.0))), rand(4, 1))
sA_check = specialize(A_check)

s1 = Scalar(0.0)
s2 = Scalar(0.0)

@finch begin
    s1 .= 0
    for j in _, i in _
        s1[] += A_check[i, j]
    end
end

@finch begin
    s2 .= 0
    for j in _, i in _
        s2[] += sA_check[i, j]
    end
end

println("Normal result:      ", s1[])
println("Specialized result: ", s2[])
println("Match:              ", s1[] ≈ s2[])
@assert s1[] ≈ s2[] "Results must match"

Normal result:      2.0482784489154375


Specialized result: 2.0482784489154375
Match:              true


## Test 5: Contrast with "Dumb Specialization" (Large N SpMV)

This is the original 100×100 SpMV example. With `N=100` columns, the outer loop is NOT eliminated (the prover can't prove `measure == 1` when shape is 100). However, the literal `100` still propagates into index calculations — the difference is **value embedding**, not structural.

This shows the boundary: specialization triggers structural changes only when shapes take **special values** (like 1) that let the prover resolve conditions.

In [8]:
let N = 100, nnz_per_col = 10
    density = nnz_per_col / N

    A = Tensor(Dense(SparseList(Element(0.0))), fsprand(N, N, density))
    x = Tensor(Dense(Element(0.0)), rand(N))
    y = Tensor(Dense(Element(0.0)))

    sA = specialize(A)

    normal_code = string(@finch_code begin
        y .= 0
        for j in _, i in _
            y[i] += A[i, j] * x[j]
        end
        return y
    end)

    specialized_code = string(@finch_code begin
        y .= 0
        for j in _, i in _
            y[i] += sA[i, j] * x[j]
        end
        return y
    end)

    # Both have the outer for-loop — shape 100 doesn't trigger loop elimination
    normal_has_loop = occursin(r"for j_\d+ = 1:", normal_code)
    spec_has_loop = occursin(r"for j_\d+ = 1:", specialized_code)

    println("N=$N SpMV — structural comparison:")
    println("  Normal has outer for-loop:      $normal_has_loop")
    println("  Specialized has outer for-loop:  $spec_has_loop")
    println()
    println("Both have the loop — no structural difference at N=$N.")
    println("The difference is only value embedding (100 vs A_lvl_stop).")
    println("This is expected: prove(ctx, 100 == 1) returns false.")
end

N=100 SpMV — structural comparison:


  Normal has outer for-loop:      true
  Specialized has outer for-loop:  true

Both have the loop — no structural difference at N=100.
The difference is only value embedding (100 vs A_lvl_stop).
This is expected: prove(ctx, 100 == 1) returns false.


## Summary

### What `specialize()` does

1. Collects Dense level shapes via `level_params(lvl)` into a type-parameter tuple
2. During virtualization, injects shapes as `literal(N)` IR nodes instead of `value(:stop, Ti)`
3. The `@staged` mechanism ensures distinct tensor shapes produce distinct code paths

### When structural simplification fires

The compiler's `prove()` function (in `src/symbolic/prove.jl`) can resolve comparisons involving only `literal` nodes. Key sites where this matters:

| Location | Condition | Effect when proved |
|---|---|---|
| `lookups.jl` | `measure(ext) == 1` | For-loop → direct assignment |
| `phases.jl` | `measure(ext) >= 0` | Guard `if stop >= start` eliminated |
| `steppers.jl` | `measure(ext) == 0` | Stepper loop simplified |
| `spikes.jl` | `measure(body_ext) <= 0` | Body portion of spike eliminated |

### Current limitations

- Only `DenseLevel` shapes are specialized; `SparseListLevel` and other levels still use runtime `value` nodes
- For typical shapes (N >> 1), the structural code is identical — only index arithmetic changes
- The `min(shape, last_stored_idx)` in SparseList phase bounds can't be eliminated because `last_stored_idx` is always a runtime unknown

### Future directions

1. **Extend `level_params` to SparseList** — specialize its shape as a literal too
2. **Structural invariants** — encode properties like "all columns non-empty" or "last index == shape" to eliminate more guards
3. **Custom level types** — levels with built-in structural guarantees that produce different looplet nests from `unfurl`